# 28 Sep
-converting the lotus json to csv


In [ ]:
import pandas as pd

df = pd.read_json('/home/school/masters/Scripts/lotus/lotus_compounds.json')
df.to_csv('/home/school/masters/Scripts/lotus/lotus_compounds.csv', index=False)


## code above didi not have the plants and I need to use that as exclusion criteria so I got perplexity to only jason me the compounds that have a plant listed as making it and no i need to make that a csv

In [3]:
import pandas as pd

df = pd.read_json('/home/school/masters/Scripts/lotus/lotus_compounds_with_plant_source.json')
df.to_csv('/home/school/masters/Scripts/lotus/lotus_compounds_with_plant_source.csv', index=False)

In [5]:
df_tot_lotus = pd.read_csv('/home/school/masters/Scripts/lotus/lotus_compounds.csv')
df_plant_lotus = pd.read_csv('/home/school/masters/Scripts/lotus/lotus_compounds_with_plant_source.csv')

tot = df_tot_lotus.shape[0]
plant = df_plant_lotus.shape[0]

print(f'Og lotus has {tot:,} compounds')
print(f'Plant lotus has {plant:,} compounds')
print(f'Percentage of plant lotus: {plant/tot*100:.2f}%')
print(f'The difference is {tot-plant:,}')

Og lotus has 276,518 compounds
Plant lotus has 192,297 compounds
Percentage of plant lotus: 69.54%
The difference is 84,221


## now for inchi deduplication

In [ ]:
import pandas as pd

df = pd.read_csv('/home/school/masters/Scripts/lotus/lotus_compounds_with_plant_source.csv')

df.drop(columns=['_id','smiles'], inplace=True)

def neat_list(values):
    if not values:
        return None
    values = [value.strip for value in values]
    seen = set()
    out_list = []
    for value in values:
        if value not in seen and pd.notna(value):
            seen.add(value)
            out_list.append(value)
    return "|".join(out_list)
    
            

df_new = df.groupby('standard_inchi',as_index=False,sort=False).agg({'lotus_id' : neat_list,
                                                                    'wikidata_id': neat_list,
                                                                    'annotation_level': neat_list,
                                                                    'np_likeness': neat_list,
                                                                    'collections': neat_list,
                                                                    'dois': neat_list,
                                                                    'synonyms': neat_list,
                                                                    'cas': neat_list,
                                                                    'coconut_id': neat_list,
                                                                    'cid': neat_list})


## So the Lotus NP people used COCONUT for their data as well so I am just going to quickly check if there are any novel compounds 

In [13]:
import pandas as pd

lotus = pd.read_csv('/home/school/masters/Scripts/lotus/lotus_compounds_with_plant_source.csv')
coconut = pd.read_csv('/home/school/masters/Scripts/coconut_full/full_coconut_for_DB.csv')

lotus_inchi = lotus['inchi']
coconut_inchi = coconut['inchi']  # not lotus['inchi']

l = set(lotus_inchi.dropna())
c = set(coconut_inchi.dropna())

if l == c:
    print('LOTUS and Coconut contain exactly the same InChIs.')
elif l.issubset(c):
    print('All LOTUS InChIs are already present in Coconut.')
else:
    missing_from_coconut = l - c
    print(f'{len(missing_from_coconut)} LOTUS InChIs are missing from Coconut.')

71667 LOTUS InChIs are missing from Coconut.


## removing all entries with nplikeness less than -1

In [3]:
import pandas as pd

lotus = pd.read_csv('/home/school/masters/Scripts/lotus/lotus_compounds_with_plant_source.csv')

og = lotus.shape[0]

lotus['npl_score'] = pd.to_numeric(lotus['npl_score'], errors='coerce')

lotus = lotus.loc[lotus['npl_score']>-1].copy()
new = lotus.shape[0]

lotus.to_csv('/home/school/masters/Scripts/lotus/lotus_compounds_with_plant_source.csv',index=False)

print(f'The old was {og:,} long and now after removing lower than -1 np likeness score it is {new:,}')


The old was 192,297 long and now after removing lower than -1 np likeness score it is 191,703


# 6 Oct


Now need to check the error file and merge the cids back to the full compounds  

In [11]:
import pandas as pd

df = pd.read_csv('/home/school/masters/Scripts/lotus/lotus_cid_errors.csv')

df = df.loc[df['error']=='PubChem HTTP Error 400 PUGREST.BadRequest: Unable to standardize the given structure - perhaps some special characters need to be escaped or data packed in a MIME form? (error: , status: 0, output: )'].copy()

compounds = df['inchi'].to_list()
print(compounds)

cid_df = pd.read_csv('/home/school/masters/Scripts/lotus/lotus_with_cid.csv')

cid_df = cid_df.loc[cid_df['inchi'].isin(compounds)].copy()

print(cid_df['inchi'].to_list())
print(cid_df['cid'].to_list())


['InChI=1S/C39H52O24/c1-11-21(43)26(48)30(52)36(56-11)55-10-19-24(46)28(50)32(54)38(61-19)63-35-29(51)23(45)18(9-40)60-39(35)62-34-25(47)20-16(42)7-15(58-37-31(53)27(49)22(44)12(2)57-37)8-17(20)59-33(34)13-3-5-14(41)6-4-13/h3-8,11-12,18-19,21-24,26-46,48-54H,9-10H2,1-2H3', 'InChI=1S/C35H64O7/c1-3-4-5-6-7-8-9-10-11-15-21-31(38)33-23-24-34(42-33)32(39)22-16-13-12-14-18-29(36)19-17-20-30(37)26-28-25-27(2)41-35(28)40/h25,27,29-34,36-39H,3-24,26H2,1-2H3', 'InChI=1S/C10H14O3/c1-7-3-4-8(9(12)5-7)10(2,13)6-11/h3-5,11-13H,6H2,1-2H3']
['InChI=1S/C39H52O24/c1-11-21(43)26(48)30(52)36(56-11)55-10-19-24(46)28(50)32(54)38(61-19)63-35-29(51)23(45)18(9-40)60-39(35)62-34-25(47)20-16(42)7-15(58-37-31(53)27(49)22(44)12(2)57-37)8-17(20)59-33(34)13-3-5-14(41)6-4-13/h3-8,11-12,18-19,21-24,26-46,48-54H,9-10H2,1-2H3', 'InChI=1S/C35H64O7/c1-3-4-5-6-7-8-9-10-11-15-21-31(38)33-23-24-34(42-33)32(39)22-16-13-12-14-18-29(36)19-17-20-30(37)26-28-25-27(2)41-35(28)40/h25,27,29-34,36-39H,3-24,26H2,1-2H3', 'InChI=1S/C10H

quickly get and add cids for those weird ones

In [10]:
import pandas as pd
import pubchempy as pcp

def get_cid(input,input_type):
    if pd.isna(input):
        return f'no input for ({input_type})'
    for k in range(5):
        try:
            compound = pcp.get_compounds(input,input_type)
            if compound:
                cid = compound[0].cid
                return cid
            else:
                return None
        except Exception as e:
            error_msg = str(e)
            if '502' in error_msg or 'bad gateway' in error_msg.lower() or 'timeout' in error_msg.lower() or '503' in error_msg or 'too many requests or server too busy' in error_msg.lower():
                time.sleep(5)
            else:
                return str(e)
    return 'retry not sucessful'

df = pd.read_csv('/home/school/masters/Scripts/lotus/lotus_cid_errors.csv')

df = df.loc[df['error']=='PubChem HTTP Error 400 PUGREST.BadRequest: Unable to standardize the given structure - perhaps some special characters need to be escaped or data packed in a MIME form? (error: , status: 0, output: )'].copy()

compounds = df['inchi'].to_list()

cid = {}
for inchi in compounds:
    cid[inchi] = get_cid(inchi,'inchi')

print(cid)

cid_df = pd.read_csv('/home/school/masters/Scripts/lotus/lotus_with_cid.csv')

for key, value in cid.items():
    cid_df.loc[cid_df['inchi'] == key, 'cid'] = value

cid_df.to_csv('/home/school/masters/Scripts/lotus/lotus_with_cid.csv',index=False)


{'InChI=1S/C39H52O24/c1-11-21(43)26(48)30(52)36(56-11)55-10-19-24(46)28(50)32(54)38(61-19)63-35-29(51)23(45)18(9-40)60-39(35)62-34-25(47)20-16(42)7-15(58-37-31(53)27(49)22(44)12(2)57-37)8-17(20)59-33(34)13-3-5-14(41)6-4-13/h3-8,11-12,18-19,21-24,26-46,48-54H,9-10H2,1-2H3': 163020850, 'InChI=1S/C35H64O7/c1-3-4-5-6-7-8-9-10-11-15-21-31(38)33-23-24-34(42-33)32(39)22-16-13-12-14-18-29(36)19-17-20-30(37)26-28-25-27(2)41-35(28)40/h25,27,29-34,36-39H,3-24,26H2,1-2H3': 85111137, 'InChI=1S/C10H14O3/c1-7-3-4-8(9(12)5-7)10(2,13)6-11/h3-5,11-13H,6H2,1-2H3': 86142411}


Adding the cids back to the full lotus

In [5]:
import pandas as pd 

lotus = pd.read_csv('/home/school/masters/Scripts/lotus/lotus_compounds_with_plant_source.csv')
cids = pd.read_csv('/home/school/masters/Scripts/lotus/lotus_with_cid.csv')

cids.drop(columns=['inchi', 'smiles', 'inchi_key'],inplace = True)

lotus = lotus.merge(cids,how='left',on='lotus_id',validate='1:1')

columns = lotus.columns

print(columns)

lotus.to_csv('/home/school/masters/Scripts/lotus/lotus_compounds_and_data_with_cids.csv',index=False)



Index(['_id', 'lotus_id', 'wikidata_id', 'smiles', 'inchi', 'traditional_name',
       'synonyms', 'iupac_name', 'collection', 'npl_score', 'cid'],
      dtype='str')


changing column names to what i need for the normalizer

In [ ]:
import pandas as pd

df = pd.read_csv('/home/school/masters/Scripts/lotus/lotus_compounds_and_data_with_cids.csv')

df.rename(columns={'traditional_name': 'n_name', 'synonyms' : 'n_synonyms', 'iupac_name': 'n_iupac'})

df.